<a href="https://colab.research.google.com/github/mushrafi88/MSE_510/blob/main/Homework_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mushrafi Munim Sushmit

0. Read Chapters 1-5 in Martin book, https://subscription.packtpub.com/book/data/9781805127161/3

1. Fitting distribution 1:
- Create a data set using 30 draws from uniform distribution on [2,5]
- Use arviz to explore the data
- Create a PYMC model fitting it with Uniform distribution
- Plot posterior distribution
- Plot posterior predictive checks
- Explain the meaning of posterior distributions

In [ ]:
# Run this cell in Google Colab to install required packages
import sys
try:
    import pymc
except ImportError:
    !pip install -q pymc arviz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import arviz as az
import pymc as pm
import pandas as pd
from sklearn.datasets import load_iris
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
az.style.use('arviz-darkgrid')
print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

In [ ]:
# --- Generate data ---
np.random.seed(RANDOM_SEED)
data1 = np.random.uniform(2, 5, 30)

print(f"n = {len(data1)}")
print(f"min  = {data1.min():.4f}")
print(f"max  = {data1.max():.4f}")
print(f"mean = {data1.mean():.4f}")
print(f"std  = {data1.std():.4f}")

In [ ]:
# --- Explore data with ArviZ ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

az.plot_kde(data1, ax=axes[0])
axes[0].set_title("Kernel Density Estimate")
axes[0].set_xlabel("Value")

axes[1].hist(data1, bins=8, color='steelblue', edgecolor='white')
axes[1].set_title("Histogram")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Count")

az.plot_ecdf(data1, ax=axes[2])
axes[2].set_title("Empirical CDF")
axes[2].set_xlabel("Value")

plt.suptitle("Problem 1: Exploring Uniform[2, 5] Data (n=30)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- PyMC model: Uniform likelihood ---
# The unknown parameters are the lower bound (a) and upper bound (b) of the
# generating Uniform distribution.  We constrain the prior so that a <= min(data)
# and b >= max(data), which ensures the likelihood is always defined.

obs_min = float(data1.min())
obs_max = float(data1.max())

with pm.Model() as model1:
    # Priors: a can be anywhere in [0, obs_min], b anywhere in [obs_max, 10]
    a = pm.Uniform('a', lower=0.0, upper=obs_min)
    b = pm.Uniform('b', lower=obs_max, upper=10.0)

    # Likelihood
    obs = pm.Uniform('obs', lower=a, upper=b, observed=data1)

    # Use Metropolis — Uniform likelihood has discontinuous gradients
    trace1 = pm.sample(
        4000, tune=2000, chains=2,
        step=pm.Metropolis(),
        random_seed=RANDOM_SEED,
        progressbar=True,
    )

print(az.summary(trace1, var_names=['a', 'b']))

In [ ]:
# --- Plot posterior distributions ---
az.plot_posterior(trace1, var_names=['a', 'b'], figsize=(12, 4))
plt.suptitle("Problem 1: Posterior Distributions of Uniform Bounds", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Posterior Predictive Checks ---
with model1:
    pm.sample_posterior_predictive(trace1, extend_inferencedata=True, random_seed=RANDOM_SEED)

az.plot_ppc(trace1, num_pp_samples=200, figsize=(10, 4))
plt.title("Problem 1: Posterior Predictive Checks")
plt.show()

The posterior distribution is basically the model’s updated understanding of what the unknown parameters are after looking at the data. In this problem, those unknown parameters are the lower bound (a) and upper bound (b) of the uniform distribution. Before seeing any samples, we only have a prior idea of where these values might lie, but once the 30 observations are included, the model updates that belief and produces the posterior. So the posterior tells us which values of (a) and (b) are most reasonable given the observed data. In the results, the posterior for (a) is concentrated near 2 and the posterior for (b) is concentrated near 5, which makes sense because the data were generated from a Uniform([2,5]) distribution. The shape of each posterior also shows uncertainty: regions with higher density are more believable, and regions with lower density are less likely. Since a uniform distribution requires all observed points to fall between (a) and (b), the smallest and largest data values strongly influence these posteriors, which is why the distributions pile up near the sample minimum and maximum.

2. Fitting distribution 2:
- Create a data set using 30 draws from uniform distribution on [2,5]
- Use arviz to explore the data
- Create a PYMC model fitting it with Beta distribution
- Plot posterior distribution
- Plot posterior predictive checks
- Explain the meaning of posterior predictive checks

In [ ]:
# Reuse data1 from Problem 1
# The Beta distribution is defined on (0, 1), so we rescale to that interval.
# We use the true generating bounds [2, 5] to avoid boundary artifacts.
data2 = (data1 - 2.0) / 3.0          # maps [2, 5] -> [0, 1]
data2 = np.clip(data2, 1e-6, 1-1e-6) # avoid exact 0 or 1

print(f"Rescaled data — min: {data2.min():.4f}, max: {data2.max():.4f}, mean: {data2.mean():.4f}")

In [ ]:
# --- Explore rescaled data with ArviZ ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

az.plot_kde(data2, ax=axes[0])
axes[0].set_title("KDE (rescaled)")
axes[0].set_xlabel("Rescaled value")

axes[1].hist(data2, bins=8, color='coral', edgecolor='white')
axes[1].set_title("Histogram (rescaled)")
axes[1].set_xlabel("Rescaled value")

az.plot_ecdf(data2, ax=axes[2])
axes[2].set_title("Empirical CDF")

plt.suptitle("Problem 2: Exploring Rescaled Data [0, 1]", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- PyMC model: Beta likelihood ---
# Beta(alpha, beta) is Uniform(0,1) when alpha=beta=1.
# Because our data truly comes from a Uniform, we expect the posterior to
# concentrate near alpha ≈ 1, beta ≈ 1.

with pm.Model() as model2:
    # Weakly informative half-Normal priors on the Beta shape parameters
    alpha = pm.HalfNormal('alpha', sigma=5.0)
    beta_param = pm.HalfNormal('beta_param', sigma=5.0)

    # Likelihood
    obs = pm.Beta('obs', alpha=alpha, beta=beta_param, observed=data2)

    trace2 = pm.sample(
        2000, tune=1000, chains=2,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
        progressbar=True,
    )

print(az.summary(trace2, var_names=['alpha', 'beta_param']))

In [ ]:
# --- Plot posterior distributions ---
az.plot_posterior(trace2, var_names=['alpha', 'beta_param'], figsize=(12, 4))
plt.suptitle("Problem 2: Posterior Distributions of Beta Shape Parameters", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Posterior Predictive Checks ---
with model2:
    pm.sample_posterior_predictive(trace2, extend_inferencedata=True, random_seed=RANDOM_SEED)

az.plot_ppc(trace2, num_pp_samples=200, figsize=(10, 4))
plt.title("Problem 2: Posterior Predictive Checks (Beta model on rescaled data)")
plt.show()

The posterior predictive check here is basically asking a simple question, after seeing the data and learning likely values of the Beta parameters, can the model generate new data that looks like what we actually observed? The many light blue lines are simulated datasets produced by the model using parameter values drawn from the posterior, so they show the range of patterns the model thinks are plausible. The black line is the real observed data, and the orange dashed line is the average prediction across those simulations. Since the black curve mostly stays within the spread of the blue curves and follows a similar overall trend, it suggests that the model is doing a decent job capturing the main structure of the data.

3. Fitting line
- Import Iris Data set
- Plot Sepal length as a function of Sepal width
- Build PYMC linear model
- Plot posteriors as marginal and joint distributions
- Plot posterior predictive checks
- Rationalize the behavior of joint distribution between slope and intersect

In [ ]:
# --- Load Iris ---
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

# Standardize predictors for numerical stability
x3_raw = df['sepal_width'].values
y3_raw = df['sepal_length'].values
x3 = (x3_raw - x3_raw.mean()) / x3_raw.std()
y3 = y3_raw  # keep response in original units

print(df.describe())

In [ ]:
# --- Scatter plot ---
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x3_raw, y3_raw, alpha=0.5, color='steelblue', edgecolors='none')
ax.set_xlabel("Sepal Width (cm)")
ax.set_ylabel("Sepal Length (cm)")
ax.set_title("Iris: Sepal Length vs. Sepal Width")
plt.tight_layout()
plt.show()

In [ ]:
# --- PyMC linear model ---
# Model:  sepal_length = intercept + slope * sepal_width_std + epsilon
#         epsilon ~ Normal(0, sigma)

with pm.Model() as model3:
    # Priors
    intercept = pm.Normal('intercept', mu=y3.mean(), sigma=2.0)
    slope     = pm.Normal('slope',     mu=0.0,       sigma=2.0)
    sigma     = pm.HalfNormal('sigma', sigma=1.0)

    # Deterministic mean
    mu = pm.Deterministic('mu', intercept + slope * x3)

    # Likelihood
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y3)

    trace3 = pm.sample(
        2000, tune=1000, chains=2,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
        progressbar=True,
    )

print(az.summary(trace3, var_names=['intercept', 'slope', 'sigma']))

In [ ]:
# --- Marginal posterior distributions ---
az.plot_posterior(trace3, var_names=['intercept', 'slope', 'sigma'], figsize=(15, 4))
plt.suptitle("Problem 3: Marginal Posterior Distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Joint posterior: slope vs. intercept ---
az.plot_pair(
    trace3,
    var_names=['intercept', 'slope'],
    kind='kde',
    divergences=True,
    figsize=(6, 6),
)
plt.suptitle("Problem 3: Joint Posterior — Slope vs. Intercept", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Posterior Predictive Checks ---
with model3:
    pm.sample_posterior_predictive(trace3, extend_inferencedata=True, random_seed=RANDOM_SEED)

az.plot_ppc(trace3, num_pp_samples=200, figsize=(10, 4))
plt.title("Problem 3: Posterior Predictive Checks")
plt.show()

# Also plot fitted line with credible band
x_range = np.linspace(x3.min(), x3.max(), 100)
post = trace3.posterior
intercept_samples = post['intercept'].values.flatten()
slope_samples     = post['slope'].values.flatten()

# Randomly select 200 posterior samples for the credible band
idx = np.random.choice(len(intercept_samples), 200, replace=False)
fig, ax = plt.subplots(figsize=(7, 5))
for i in idx:
    ax.plot(x_range * x3_raw.std() + x3_raw.mean(),
            intercept_samples[i] + slope_samples[i] * x_range,
            color='orange', alpha=0.05, linewidth=0.8)
ax.scatter(x3_raw, y3_raw, s=20, color='steelblue', zorder=5, alpha=0.7)
ax.set_xlabel("Sepal Width (cm)")
ax.set_ylabel("Sepal Length (cm)")
ax.set_title("Problem 3: Posterior Predictive Lines (200 samples)")
plt.tight_layout()
plt.show()

The joint posterior between the slope and intercept shows how these two parameters trade off against each other while still fitting the data reasonably well. In this plot, the high-density region is centered around an intercept of about 5.84 and a slope of about −0.10, which matches the marginal summaries. The shape is fairly round to slightly tilted, meaning there is only a weak dependence between the two parameters.

4. Fitting plane
- Import Iris Data set
- Plot Sepal length as a function of Petal Length and Petal width
- Build PYMC linear model
- Plot posteriors
- Plot posterior predictive checks

In [ ]:
# --- Prepare data ---
x4a_raw = df['petal_length'].values
x4b_raw = df['petal_width'].values
y4_raw  = df['sepal_length'].values

# Standardize predictors
x4a = (x4a_raw - x4a_raw.mean()) / x4a_raw.std()
x4b = (x4b_raw - x4b_raw.mean()) / x4b_raw.std()
y4  = y4_raw

In [ ]:
# --- 3-D scatter plot ---
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(x4a_raw, x4b_raw, y4_raw,
                     c=y4_raw, cmap='viridis', alpha=0.7, s=25)
ax.set_xlabel("Petal Length (cm)")
ax.set_ylabel("Petal Width (cm)")
ax.set_zlabel("Sepal Length (cm)")
ax.set_title("Iris: Sepal Length vs. Petal Dimensions")
fig.colorbar(scatter, ax=ax, label="Sepal Length")
plt.show()

# Also show marginal scatter plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x4a_raw, y4_raw, alpha=0.5, color='steelblue')
axes[0].set_xlabel("Petal Length (cm)")
axes[0].set_ylabel("Sepal Length (cm)")
axes[0].set_title("Sepal Length vs. Petal Length")
axes[1].scatter(x4b_raw, y4_raw, alpha=0.5, color='coral')
axes[1].set_xlabel("Petal Width (cm)")
axes[1].set_ylabel("Sepal Length (cm)")
axes[1].set_title("Sepal Length vs. Petal Width")
plt.tight_layout()
plt.show()

In [ ]:
# --- PyMC multiple linear regression model ---
# sepal_length = intercept + b1 * petal_length_std + b2 * petal_width_std + epsilon

with pm.Model() as model4:
    intercept = pm.Normal('intercept', mu=y4.mean(), sigma=2.0)
    b1        = pm.Normal('b_petal_length', mu=0.0, sigma=2.0)
    b2        = pm.Normal('b_petal_width',  mu=0.0, sigma=2.0)
    sigma     = pm.HalfNormal('sigma', sigma=1.0)

    mu = pm.Deterministic('mu', intercept + b1 * x4a + b2 * x4b)

    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y4)

    trace4 = pm.sample(
        2000, tune=1000, chains=2,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
        progressbar=True,
    )

print(az.summary(trace4, var_names=['intercept', 'b_petal_length', 'b_petal_width', 'sigma']))

In [ ]:
# --- Marginal posteriors ---
az.plot_posterior(trace4, var_names=['intercept', 'b_petal_length', 'b_petal_width', 'sigma'],
                  figsize=(16, 4))
plt.suptitle("Problem 4: Posterior Distributions — Plane Regression", y=1.02)
plt.tight_layout()
plt.show()

# Joint posteriors for the two slope coefficients
az.plot_pair(
    trace4,
    var_names=['b_petal_length', 'b_petal_width'],
    kind='kde',
    figsize=(5, 5),
)
plt.suptitle("Joint Posterior: Petal Length vs. Petal Width Coefficients", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Posterior Predictive Checks ---
with model4:
    pm.sample_posterior_predictive(trace4, extend_inferencedata=True, random_seed=RANDOM_SEED)

az.plot_ppc(trace4, num_pp_samples=200, figsize=(10, 4))
plt.title("Problem 4: Posterior Predictive Checks — Plane Regression")
plt.show()

# Observed vs. posterior predictive mean
y_pred_mean = trace4.posterior_predictive['y_obs'].values.mean(axis=(0, 1))
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y4, y_pred_mean, alpha=0.6, color='steelblue')
ax.plot([y4.min(), y4.max()], [y4.min(), y4.max()], 'r--', label='y = ŷ')
ax.set_xlabel("Observed Sepal Length")
ax.set_ylabel("Posterior Predictive Mean")
ax.set_title("Problem 4: Observed vs. Predicted")
ax.legend()
plt.tight_layout()
plt.show()

5. Build Hierarchical model
- Import Iris data set
- Fit Sepal Length as a function of Sepal Width for three groups as hierarchical model
- Explain the meanings of the hyperparameters proprs and parameters priors

In [ ]:
# --- Prepare data ---
species_names = iris.target_names            # ['setosa', 'versicolor', 'virginica']
species_idx   = df['species'].values         # integer codes 0, 1, 2
n_species     = len(np.unique(species_idx))

x5_raw = df['sepal_width'].values
y5_raw = df['sepal_length'].values

# Standardize predictor (pooled)
x5 = (x5_raw - x5_raw.mean()) / x5_raw.std()
y5 = y5_raw

# Quick look at the three species
colors = ['steelblue', 'coral', 'forestgreen']
fig, ax = plt.subplots(figsize=(7, 5))
for i, name in enumerate(species_names):
    mask = species_idx == i
    ax.scatter(x5_raw[mask], y5_raw[mask], label=name, color=colors[i], alpha=0.7)
ax.set_xlabel("Sepal Width (cm)")
ax.set_ylabel("Sepal Length (cm)")
ax.set_title("Iris: Sepal Length vs. Sepal Width by Species")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Hierarchical PyMC model ---
#
# Each species s gets its own intercept a_s and slope b_s.
# Instead of fitting them independently (no-pooling) or collapsing them
# (complete pooling), partial pooling allows the species parameters to
# share information via group-level (hyper) distributions.
#
# Hyperparameters (population level):
#   mu_a    ~ Normal(mean(y), 2)   — population mean intercept
#   sigma_a ~ HalfNormal(1)        — population SD of intercepts
#   mu_b    ~ Normal(0, 1)         — population mean slope
#   sigma_b ~ HalfNormal(1)        — population SD of slopes
#
# Parameters (species level):
#   a_s ~ Normal(mu_a, sigma_a)    — species-specific intercept
#   b_s ~ Normal(mu_b, sigma_b)    — species-specific slope
#
# Likelihood:
#   y_i ~ Normal(a_{s(i)} + b_{s(i)} * x_i, sigma_obs)

with pm.Model() as model5:
    # --- Hyperpriors (population-level) ---
    mu_a    = pm.Normal('mu_a',    mu=y5.mean(), sigma=2.0)
    sigma_a = pm.HalfNormal('sigma_a', sigma=1.0)

    mu_b    = pm.Normal('mu_b',    mu=0.0, sigma=1.0)
    sigma_b = pm.HalfNormal('sigma_b', sigma=1.0)

    # --- Species-level parameters (drawn from the population distributions) ---
    a = pm.Normal('a', mu=mu_a, sigma=sigma_a, shape=n_species)
    b = pm.Normal('b', mu=mu_b, sigma=sigma_b, shape=n_species)

    # --- Observation noise ---
    sigma_obs = pm.HalfNormal('sigma_obs', sigma=1.0)

    # --- Likelihood ---
    mu = pm.Deterministic('mu', a[species_idx] + b[species_idx] * x5)
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma_obs, observed=y5)

    trace5 = pm.sample(
        2000, tune=2000, chains=2,
        target_accept=0.95,
        random_seed=RANDOM_SEED,
        progressbar=True,
    )

print(az.summary(trace5, var_names=['mu_a', 'sigma_a', 'mu_b', 'sigma_b', 'a', 'b', 'sigma_obs']))

In [ ]:
# --- Posterior: hyperparameters ---
az.plot_posterior(trace5, var_names=['mu_a', 'sigma_a', 'mu_b', 'sigma_b'], figsize=(16, 4))
plt.suptitle("Problem 5: Posteriors — Hyperparameters (Population Level)", y=1.02)
plt.tight_layout()
plt.show()

# --- Posterior: species-level parameters ---
az.plot_posterior(trace5, var_names=['a', 'b'], figsize=(16, 8))
plt.suptitle("Problem 5: Posteriors — Species Intercepts (a) and Slopes (b)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Forest plot: species intercepts and slopes ---
az.plot_forest(trace5, var_names=['a', 'b'], combined=True,
               figsize=(8, 5), hdi_prob=0.94)
plt.title("Problem 5: Forest Plot — Species-Level Parameters")
plt.tight_layout()
plt.show()

In [ ]:
# --- Posterior Predictive Checks ---
with model5:
    pm.sample_posterior_predictive(trace5, extend_inferencedata=True, random_seed=RANDOM_SEED)

az.plot_ppc(trace5, num_pp_samples=200, figsize=(10, 4))
plt.title("Problem 5: Posterior Predictive Checks — Hierarchical Model")
plt.show()

# Plot fitted lines per species
x_range_std = np.linspace(x5.min(), x5.max(), 100)
x_range_raw = x_range_std * x5_raw.std() + x5_raw.mean()

post5 = trace5.posterior
fig, ax = plt.subplots(figsize=(8, 6))
for i, (name, color) in enumerate(zip(species_names, colors)):
    mask = species_idx == i
    ax.scatter(x5_raw[mask], y5_raw[mask], color=color, alpha=0.5, label=name)

    # Posterior mean line
    a_mean = float(post5['a'].values[:, :, i].mean())
    b_mean = float(post5['b'].values[:, :, i].mean())
    ax.plot(x_range_raw, a_mean + b_mean * x_range_std,
            color=color, linewidth=2.5)

ax.set_xlabel("Sepal Width (cm)")
ax.set_ylabel("Sepal Length (cm)")
ax.set_title("Problem 5: Hierarchical Model — Posterior Mean Lines per Species")
ax.legend()
plt.tight_layout()
plt.show()

In a hierarchical model, parameter priors are the priors placed on the actual parameters for each group, such as the intercept and slope for each iris species, while hyperparameter priors are the priors placed on the higher-level quantities that control how those group-specific parameters are distributed overall. In this case, the species-specific intercepts and slopes are the parameters, and the overall mean and spread that govern them are the hyperparameters.